In [1]:
from datetime import datetime, timedelta
from utils.spark_session import get_sparkSession 
from utils.utilities import generate_date

In [2]:
from pyspark.sql.functions import current_timestamp, lit, expr, to_date, when, lower, upper, trim

In [3]:
spark = get_sparkSession("Load silver.fact_ads_performance")

In [4]:
print("SPARK-APP : spark-UI - " + spark.sparkContext.uiWebUrl)

SPARK-APP : spark-UI - http://ba7b2bbb9aec:4040


In [5]:
from utils.load_controller import insert_control_logs, get_max_loadTimestamp

In [6]:
_schema_name = "silver"
_table_name = "fact_ads_performance"
_fullname = f"{_schema_name}.{_table_name}"
_script_name = "silver_fact_ads_performance.ipynb"
_bronze_table = "bronze.fact_ads_performance"

print(f"SPARK-APP : Loading started for {_fullname}")

SPARK-APP : Loading started for silver.fact_ads_performance


In [7]:
#spark config

spark.conf.set("spark.sql.shuffle.partitions",16)


In [8]:
#Read get_max_loadTimestamp

max_timestamp = get_max_loadTimestamp(spark, _schema_name, _table_name)


In [9]:
#Reading from bronze.raw_customer and de-duplicating the data

df = spark.read.table(_bronze_table) \
          .where(f"created_on > to_timestamp('{max_timestamp}')")

print(f"SPARK-APP: Bronze Table Data count : {df.count()}")

df_dedup = df.withColumn("_rn", expr("row_number() over (partition by campaign_id, ads_date, store, channel sort by created_on desc)")) \
             .where("_rn = 1") \
             .drop("_rn")

loaded_count = df_dedup.count()

print(f"SPARK-APP: Table count after de-dupe : {loaded_count}")

SPARK-APP: Bronze Table Data count : 1
SPARK-APP: Table count after de-dupe : 1


In [10]:
df_dedup.show(10)
df_dedup.printSchema()


+-----------+--------+------+-------+---------------+-----------+------+----+--------+----------------+-----------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|campaign_id|ads_date| store|channel|  campaign_name|impressions|clicks|cost|currency|orders_generated|revenue_generated|          created_on|          created_by|         modified_on|         modified_by|         source_file|
+-----------+--------+------+-------+---------------+-----------+------+----+--------+----------------+-----------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|     AD1001|20250101|AMZ_US|    MKT|Amazon Campaign|       1000|    50| 200|     USD|               5|             2250|2026-01-29 00:27:...|raw_fact_ads_perf...|2026-01-29 00:27:...|raw_fact_ads_perf...|fact_ads_performa...|
+-----------+--------+------+-------+---------------+-----------+------+----+--------+------

In [11]:
#Formating the data

df_ld = df_dedup.withColumn("ads_date", to_date("ads_date", 'yyyyMMdd')) \
                .withColumn("cost", expr("CAST(cost as decimal(18,2))")) \
                .withColumn("impressions", expr("CAST(impressions as int)")) \
                .withColumn("clicks", expr("CAST(clicks as int)")) \
                .withColumn("orders_generated", expr("CAST(orders_generated as int)")) \
                .withColumn("revenue_generated", expr("CAST(revenue_generated as decimal(18,2))"))


In [12]:
#Standardizing the codes and types

df_ld = df_ld.withColumn("campaign_id", upper(trim("campaign_id"))) \
             .withColumn("store", upper(trim("store"))) \
             .withColumn("channel", upper(trim("channel"))) \
             .withColumn("currency", upper(trim("currency")))


In [13]:
#Adding audit columns

df_ld = df_ld.withColumn("created_on", current_timestamp()) \
             .withColumn("created_by", lit(_script_name)) \
             .withColumn("modified_on", current_timestamp()) \
             .withColumn("modified_by", lit(_script_name))

In [14]:
df_ld.show(10)
df_ld.printSchema()

+-----------+----------+------+-------+---------------+-----------+------+------+--------+----------------+-----------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|campaign_id|  ads_date| store|channel|  campaign_name|impressions|clicks|  cost|currency|orders_generated|revenue_generated|          created_on|          created_by|         modified_on|         modified_by|         source_file|
+-----------+----------+------+-------+---------------+-----------+------+------+--------+----------------+-----------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|     AD1001|2025-01-01|AMZ_US|    MKT|Amazon Campaign|       1000|    50|200.00|     USD|               5|          2250.00|2026-01-29 00:56:...|silver_fact_ads_p...|2026-01-29 00:56:...|silver_fact_ads_p...|fact_ads_performa...|
+-----------+----------+------+-------+---------------+-----------+------+--

In [15]:
#Writing to Table and log data to load_controller

_data = []

df_ld.write \
     .format("delta") \
     .mode("overwrite") \
     .saveAsTable(_fullname)
    
_data.append([_schema_name, _schema_name, _table_name, "full-load", "overwrite", str(datetime.now()), "success", loaded_count, _script_name, _script_name])

insert_control_logs(spark, _data)
    
print(f"SPARK-APP: Data written to {_schema_name}.{_table_name} and load_controller")

SPARK-APP: Data written to silver.fact_ads_performance and load_controller


In [16]:
spark.read.table(f"{_schema_name}.{_table_name}").show()

+-----------+----------+------+-------+---------------+-----------+------+------+--------+----------------+-----------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|campaign_id|  ads_date| store|channel|  campaign_name|impressions|clicks|  cost|currency|orders_generated|revenue_generated|          created_on|          created_by|         modified_on|         modified_by|         source_file|
+-----------+----------+------+-------+---------------+-----------+------+------+--------+----------------+-----------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|     AD1001|2025-01-01|AMZ_US|    MKT|Amazon Campaign|       1000|    50|200.00|     USD|               5|          2250.00|2026-01-29 00:56:...|silver_fact_ads_p...|2026-01-29 00:56:...|silver_fact_ads_p...|fact_ads_performa...|
+-----------+----------+------+-------+---------------+-----------+------+--

In [17]:
spark.sql(f"select * from gold.load_controller where schema_name = '{_schema_name}' and table_name = '{_table_name}' order by created_on desc").show()

+------+-----------+--------------------+---------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+
| layer|schema_name|          table_name|load_type|write_mode|      load_timestamp|load_status|loaded_count|          created_on|          created_by|         modified_on|         modified_by|
+------+-----------+--------------------+---------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+
|silver|     silver|fact_ads_performance|full-load| overwrite|2026-01-29 00:56:...|    success|           1|2026-01-29 00:56:...|silver_fact_ads_p...|2026-01-29 00:56:...|silver_fact_ads_p...|
+------+-----------+--------------------+---------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+



In [18]:
from delta import DeltaTable

dt = DeltaTable.forName(spark,f"{_schema_name}.{_table_name}")

dt.history().limit(1).select("version","operationMetrics.numOutputRows").show(truncate=False)

+-------+-------------+
|version|numOutputRows|
+-------+-------------+
|0      |1            |
+-------+-------------+



In [19]:
spark.stop()